# TAD importance genomic validation

In [ ]:
mode = "subset"


In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

colors = [
    "#70acc0",
    "#c12075",
    "#003754",
    "#4c626c",
    "#be9e7c",
    "#723c5d",
    "#9d9c9c",
    "#00695c",
    "#e57373",
]
colors_name = [
    "graublau",
    "purpur",
    "blau",
    "dunkelgrau",
    "bronze",
    "violett",
    "hellgrau",
    "gedämpftes teal",
    "sanftes koralle",
]
sns.set_palette(sns.color_palette(colors))
base_color = "#70acc0"
light_palette = sns.light_palette(base_color, as_cmap=False)
colors = light_palette + ["#4c626c", "#003754"]
cmap = LinearSegmentedColormap.from_list("custom_cmap", colors)
colors_heat = ["#c12075", "#f0f0f0", "#003754"]
cmap_heat = LinearSegmentedColormap.from_list("custom_diverging", colors_heat, N=256)
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"

import re as _re
import matplotlib.figure as _mpl_figure

def _fix_svg_font_declarations(_svg_path):
    """Convert matplotlib's unsupported CSS 'font:' shorthand (labeled px, but
    actually point-sized, in a pt-calibrated viewBox) into explicit
    font-family/font-size/font-weight/font-style, which is what Figma's SVG
    importer requires to resolve size and style correctly."""
    try:
        with open(_svg_path, "r", encoding="utf-8") as _f:
            _content = _f.read()
    except OSError:
        return
    def _convert(_m):
        _oblique = _m.group(1)
        _weight = _m.group(2)
        _pt = float(_m.group(3)) * 0.75
        _family = _m.group(4)
        _font_weight = "bold" if _weight else "normal"
        _font_style = "oblique" if _oblique else "normal"
        return (
            f"font-family: {_family}; font-size: {_pt:g}pt; "
            f"font-weight: {_font_weight}; font-style: {_font_style}"
        )
    _pattern = _re.compile(r'font:\s*(oblique\s+)?(?:(\d+)\s+)?([0-9.]+)px\s+([^;"]+)')
    _new_content = _pattern.sub(_convert, _content)
    if _new_content != _content:
        with open(_svg_path, "w", encoding="utf-8") as _f:
            _f.write(_new_content)

_orig_figure_savefig = _mpl_figure.Figure.savefig

def _rasterize_dense_elements(_fig):
    for _ax in _fig.get_axes():
        for _coll in list(_ax.collections):
            try:
                _coll.set_rasterized(True)
            except Exception:
                pass
        for _im in _ax.get_images():
            try:
                _im.set_rasterized(True)
            except Exception:
                pass

def _patched_figure_savefig(self, fname, *args, **kwargs):
    if isinstance(fname, str) and fname.lower().endswith(".svg"):
        _rasterize_dense_elements(self)
        kwargs.setdefault("dpi", 300)
    _result = _orig_figure_savefig(self, fname, *args, **kwargs)
    if isinstance(fname, str) and fname.lower().endswith(".svg"):
        _fix_svg_font_declarations(fname)
    return _result

_mpl_figure.Figure.savefig = _patched_figure_savefig
mpl.rcParams["figure.dpi"] = 500.0
mpl.rcParams["savefig.transparent"] = True
mpl.rcParams["savefig.format"] = "png"

mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["DejaVu Sans"]

mpl.rcParams["axes.edgecolor"] = "black"
mpl.rcParams["axes.linewidth"] = 0.8
mpl.rcParams["axes.grid"] = True
mpl.rcParams["grid.color"] = "grey"
mpl.rcParams["grid.linestyle"] = "--"
mpl.rcParams["grid.linewidth"] = 0.5

mpl.rcParams["legend.frameon"] = False
mpl.rcParams["legend.loc"] = "best"

mpl.rcParams["xtick.major.size"] = 3
mpl.rcParams["xtick.minor.size"] = 2
mpl.rcParams["ytick.major.size"] = 3
mpl.rcParams["ytick.minor.size"] = 2
mpl.rcParams["xtick.major.width"] = 0.4
mpl.rcParams["xtick.minor.width"] = 0.3
mpl.rcParams["ytick.major.width"] = 0.4
mpl.rcParams["ytick.minor.width"] = 0.3
scaling_factor = 1
if mode == "all":
    base_font_size = 4
else:
    base_font_size = 6
base_axes_title_size = base_font_size + 1
base_axes_label_size = base_font_size
base_tick_label_size = base_font_size - 0.5
base_legend_font_size = base_font_size - 0.5
base_figure_title_size = base_font_size + 2
mpl.rcParams["font.size"] = base_font_size * scaling_factor
mpl.rcParams["axes.titlesize"] = base_axes_title_size * scaling_factor
mpl.rcParams["axes.labelsize"] = base_axes_label_size * scaling_factor
mpl.rcParams["xtick.labelsize"] = base_tick_label_size * scaling_factor
mpl.rcParams["ytick.labelsize"] = base_tick_label_size * scaling_factor
mpl.rcParams["legend.fontsize"] = base_legend_font_size * scaling_factor
mpl.rcParams["figure.titlesize"] = base_figure_title_size * scaling_factor

mpl.rcParams["patch.linewidth"] = 0.7

mpl.rcParams["figure.autolayout"] = True


In [ ]:
def set_figure_params(column, height):

    single_column_width = 3.5
    double_column_width = 7.0

    if column == "single":
        figure_width = single_column_width
    elif column == "double":
        figure_width = double_column_width
    else:
        figure_width = column
    scaling_factor = figure_width / single_column_width
    scaling_factor = 1

    mpl.rcParams["figure.figsize"] = (figure_width, height)


In [ ]:
import sys
import os

_OUTDIR_ABS = os.path.join(os.getcwd(), "outputs", "figS19_S23_neurips2021_cite_bmmc_fc_interaction_gene_layer_genomic_validation")
os.makedirs(_OUTDIR_ABS, exist_ok=True)

sys.path.append(os.path.abspath("../.experiment_data/primary/benchmarking"))

import contextlib
import functools

_POSTPROC_CWD = os.getcwd()

@contextlib.contextmanager
def _in_primary_dir():
    os.chdir(os.path.join(_POSTPROC_CWD, "../.experiment_data/primary"))
    try:
        yield
    finally:
        os.chdir(_POSTPROC_CWD)

with _in_primary_dir():
    import muon as _muon
    _orig_sample_obs = _muon.pp.sample_obs
    def _patched_sample_obs(data, *a, **kw):
        return _orig_sample_obs(data, *a, **kw).copy()
    _muon.pp.sample_obs = _patched_sample_obs
    import datasets

for _name in dir(datasets):
    if _name.startswith("load_"):
        def _make_wrapper(fn):
            @functools.wraps(fn)
            def _wrapper(*a, **kw):
                with _in_primary_dir():
                    return fn(*a, **kw)
            return _wrapper
        setattr(datasets, _name, _make_wrapper(getattr(datasets, _name)))
import glob
import pandas as pd
import scipy
import numpy as np
import os
from tqdm import tqdm
import pickle
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, pearsonr
from scipy.spatial.distance import euclidean
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pickle
import glob
from collections import Counter
from pathlib import Path
from goatools.obo_parser import GODag
from goatools.anno.gaf_reader import GafReader
import glob
import pandas as pd
import scipy
import numpy as np
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap
from collections import defaultdict
import sklearn
from pysankey import sankey
from sklearn.preprocessing import normalize
import pandas as pd
import numpy as np
import scipy
import matplotlib.pyplot as plt
import os
import pyBigWig
from scipy.stats import spearmanr
import ast
import glob
from pathlib import Path
import math

os.makedirs("outputs/figS19_S23_neurips2021_cite_bmmc_fc_interaction_gene_layer_genomic_validation", exist_ok=True)


In [ ]:
phenotypes = [
    "HSC",
    "Lymph_prog",
    "G_M_prog",
    "MK_E_prog",
    "ILC",
    "ILC1",
    "NK",
    "NK_CD158e1+",
    "CD4+_T_naive",
    "CD4+_T_activated",
    "CD4+_T_activated_integrinB7+",
    "CD4+_T_CD314+_CD45RA+",
    "CD8+_T_naive",
    "CD8+_T_naive_CD127+_CD26-_CD101-",
    "T_prog_cycling",
    "CD8+_T_CD49f+",
    "CD8+_T_CD69+_CD45RA+",
    "CD8+_T_CD69+_CD45RO+",
    "CD8+_T_TIGIT+_CD45RA+",
    "CD8+_T_TIGIT+_CD45RO+",
    "CD8+_T_CD57+_CD45RA+",
    "CD8+_T_CD57+_CD45RO+",
    "gdT_CD158b+",
    "gdT_TCRVD2+",
    "dnT",
    "T_reg",
    "Naive_CD20+_B_IGKC+",
    "Naive_CD20+_B_IGKC-",
    "Transitional_B",
    "B1_B_IGKC+",
    "B1_B_IGKC-",
    "Plasma_cell_IGKC+",
    "Plasma_cell_IGKC-",
    "Plasmablast_IGKC+",
    "Plasmablast_IGKC-",
    "cDC1",
    "cDC2",
    "pDC",
    "CD14+_Mono",
    "CD16+_Mono",
    "Erythroblast",
    "Normoblast",
    "Proerythroblast",
    "Reticulocyte",
    "MAIT",
]
phenotypes_labels = [
    "HSC",
    "Lymph prog",
    "G/M prog",
    "MK/E prog",
    "ILC",
    "ILC1",
    "NK",
    "NK CD158e1+",
    "CD4+ T naive",
    "CD4+ T activated",
    "CD4+ T activated B7+",
    "CD4+ T CD314+ CD45RA+",
    "CD8+ T naive",
    "CD8+ T naive CD127+ CD26- CD101-",
    "T prog cycling",
    "CD8+ T CD49f+",
    "CD8+ T CD69+ CD45RA+",
    "CD8+ T CD69+ CD45RO+",
    "CD8+ T TIGIT+ CD45RA+",
    "CD8+ T TIGIT+ CD45RO+",
    "CD8+ T CD57+ CD45RA+",
    "CD8+ T CD57+ CD45RO+",
    "gdT CD158b+",
    "gdT TCRVD2+",
    "dnT",
    "T reg",
    "Naive CD20+ B IGKC+",
    "Naive CD20+ B IGKC-",
    "Transitional B",
    "B1 B IGKC+",
    "B1 B IGKC-",
    "Plasma cell IGKC+",
    "Plasma cell IGKC-",
    "Plasmablast IGKC+",
    "Plasmablast IGKC-",
    "cDC1",
    "cDC2",
    "pDC",
    "CD14+ Mono",
    "CD16+ Mono",
    "Erythroblast",
    "Normoblast",
    "Proerythroblast",
    "Reticulocyte",
    "MAIT",
]
phenotypes_dataset = phenotypes_labels


In [ ]:
import pandas as pd

_GENE_POSITION_MAPPING_PATH = os.path.abspath(
    os.path.join(
        "..", "preprocessing", "resources", "ensembl_gene_id_chromosome_position_mapping",
        "ensembl_gene_id_chromosome_position_mapping.csv",
    )
)

def _get_ensembl_gene_id_chromosome_name_start_position_end_position_mapping():
    return pd.read_csv(_GENE_POSITION_MAPPING_PATH)


In [ ]:
def aggregate_dataframes(graph_csv_list):
    num_rows = len(graph_csv_list[0])
    for df in graph_csv_list:
        if len(df) != num_rows:
            raise ValueError("Alle DataFrames müssen die gleiche Anzahl an Zeilen haben.")

    aggregated_roc = []
    aggregated_activations = []

    for row_idx in range(num_rows):

        roc_values = [df.iloc[row_idx]["predictors_label_valid_roc"] for df in graph_csv_list]

        mean_roc = np.mean(roc_values, axis=0)
        aggregated_roc.append(mean_roc)

        activations_arrays = [df.iloc[row_idx]["activations_valid"] for df in graph_csv_list]

        concatenated_activations = np.concatenate(activations_arrays, axis=1)

        mean_activations = np.mean(concatenated_activations, axis=1)
        aggregated_activations.append(mean_activations)

    aggregated_df = pd.DataFrame(
        index=graph_csv_list[0].index,
        data={
            "predictors_label_valid_roc": aggregated_roc,
            "activations_valid": aggregated_activations,
        },
    )

    return aggregated_df


In [ ]:
seeds = ["0"]

graph_csv_list = []
graph_csv_index_list = []
gene_groups_importances = []
for modality in tqdm(["expression", "protein"]):
    gene_groups_importances_modality = []
    graph_csv_modality_list = []
    for phenotype in tqdm([x for _, x in sorted(zip(phenotypes_labels, phenotypes))]):
        graph_csv_phenotypes_list = []
        graph_csv = pd.read_pickle(
            f"../.experiment_data/scratch/experiments_new/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_moeTrue_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/calc_gene_groups_cell_type/{modality}_cell_type_vs_rest_{phenotype}_res_unp_act_mask_False_flip_False_gene_groups_ENCFF173VDJ_K562_GRCh38_contact_domains_juicertools_gene_groups_evaluated_graph_csv.pkl"
        )
        graph_csv.index = ["_".join(sorted(val.split("_"))) for val in graph_csv.index]
        graph_csv = graph_csv.loc[~graph_csv.index.duplicated(keep="first")]
        graph_csv_phenotypes_list.append(graph_csv)
        graph_csv = aggregate_dataframes(graph_csv_phenotypes_list)
        graph_csv_modality_list.append(graph_csv)
        gene_groups_importances_modality.append(np.array(graph_csv["predictors_label_valid_roc"]))
    graph_csv_list.append(graph_csv_modality_list)
    graph_csv_index_list.append(np.array(graph_csv_modality_list[0].index))
    gene_groups_importances.append(gene_groups_importances_modality)


In [ ]:
seeds = ["0"]

graph_csv_baseline_list = []
graph_csv_index_baseline_list = []
gene_groups_baseline_importances = []
for modality in tqdm(["expression", "protein"]):
    gene_groups_importances_baseline_modality = []
    graph_csv_modality_baseline_list = []
    for phenotype in tqdm([x for _, x in sorted(zip(phenotypes_labels, phenotypes))]):
        graph_csv_phenotypes_baseline_list = []
        graph_csv = pd.read_pickle(
            f"../.experiment_data/scratch/experiments/sparsevi_neurips2021_cite_bmmc_1.0_scib_hvar_True_batscib_True_patscib_False_scarches_False_ec_True_dic_True_ddic_True_batcov_True_patcov_False_modtype_go_with_gene_layer_genelayerint_ENCFF173VDJ_gaf_standard_with_rna_s0/default/calc_gene_groups_cell_type/{modality}_cell_type_vs_rest_{phenotype}_res_unp_act_mask_False_flip_False_gene_groups_ENCFF173VDJ_K562_GRCh38_contact_domains_juicertools_gene_groups_evaluated_graph_csv.pkl"
        )
        graph_csv.index = ["_".join(sorted(val.split("_"))) for val in graph_csv.index]
        graph_csv = graph_csv.loc[~graph_csv.index.duplicated(keep="first")]
        graph_csv_phenotypes_baseline_list.append(graph_csv)
        graph_csv = aggregate_dataframes(graph_csv_phenotypes_baseline_list)
        graph_csv_modality_baseline_list.append(graph_csv)
        gene_groups_importances_baseline_modality.append(
            np.array(graph_csv["predictors_label_valid_roc"])
        )
    graph_csv_baseline_list.append(graph_csv_modality_baseline_list)
    graph_csv_index_baseline_list.append(np.array(graph_csv_modality_baseline_list[0].index))
    gene_groups_baseline_importances.append(gene_groups_importances_baseline_modality)


# Gene groups importance statistics

In [ ]:
len(np.where(np.array(gene_groups_importances[0]).flatten() > 0.65)[0])


In [ ]:
len(np.where(np.array(gene_groups_importances[0]).flatten() > 0.6)[0])


In [ ]:
len(np.where(np.array(gene_groups_importances[0]).flatten() > 0.9)[0])


In [ ]:
os.getcwd()


In [ ]:
for modality_index, modality in tqdm(enumerate(['expression', 'protein'])):
    set_figure_params('double', 9)
    fig, axes = plt.subplots(9, 5)
    axes = axes.flatten()
    for i, (ax, _) in enumerate(zip(axes, gene_groups_importances[modality_index])):
        ax.hist(gene_groups_importances[modality_index][i], bins=50)
        ax.set_title(sorted(phenotypes_labels)[i])
        ax.set_xlabel('GO importance')
        ax.set_ylabel('prevalence')
        ax.set_xlim(right=1.0)
    plt.tight_layout()
    plt.show()


# Biological markers correlation

In [ ]:
# ENCODE bigWig tracks
paths_genomic_markers = glob.glob(
    os.path.join(os.path.abspath("../.experiment_data/scratch/../encode"), "*_K562_*.bigWig")
)


In [ ]:
not_calc_encode_ids = (
    []
)


In [ ]:
paths_genomic_markers = [
    path
    for path in paths_genomic_markers
    if not any([not_calc_encode_id in path for not_calc_encode_id in not_calc_encode_ids])
]


In [ ]:
genomic_markers = [
    str(Path(path_genomic_marker).stem).split("_")[2]
    for path_genomic_marker in paths_genomic_markers
]


In [ ]:
paths_genomic_markers = [x for _, x in sorted(zip(genomic_markers, paths_genomic_markers))]
genomic_markers = sorted(genomic_markers)


In [ ]:
analysis_labs = ["michael snyder", "richard myers", "xiang dong fu", "bradley bernstein"]


In [ ]:
import requests

def get_lab(path):

    file = path.split("/")[-1].split("_")[0]

    url = f"https://www.encodeproject.org/{file}?frame=object"
    headers = {"accept": "application/json"}
    response = requests.get(url, headers=headers)
    biosample = response.json()

    file = biosample["dataset"].split("/")[-2]

    url = f"https://www.encodeproject.org/{file}?frame=object"
    headers = {"accept": "application/json"}
    response = requests.get(url, headers=headers)
    biosample = response.json()

    return biosample["lab"].split("/")[-2].replace("-", " ")


In [ ]:
labs = [get_lab(path) for path in tqdm(paths_genomic_markers)]


In [ ]:
for encff in [
    Path(path_genomic_marker).stem.split("_")[0]
    for path_genomic_marker, lab in zip(paths_genomic_markers, labs)
    if lab == "michael snyder"
]:
    print(encff)


In [ ]:
selected_genomic_markers_positions = []
selected_genomic_markers_labels = []

for analysis_lab in analysis_labs:
    selected_genomic_markers_positions.append(
        [
            genomic_marker_position
            for genomic_marker_position, (genomic_marker, lab) in enumerate(
                zip(genomic_markers, labs)
            )
            if lab == analysis_lab
        ]
    )
    selected_genomic_markers_labels.append(
        [
            genomic_marker
            for genomic_marker_position, (genomic_marker, lab) in enumerate(
                zip(genomic_markers, labs)
            )
            if lab == analysis_lab
        ]
    )


In [ ]:
def get_corr_go_importance_genomic_markers(
    graph_csv_index_list, gene_groups_importances, paths_genomic_markers, bw_type="max"
):

    genomic_marker_corr_modalities = []
    genomic_marker_corr_p_modalities = []

    for modality_index in list(range(len(graph_csv_index_list))):
        gene_groups_not_found_bw = []
        genomic_marker_vals = []

        for gene_group_index, gene_group in tqdm(enumerate(graph_csv_index_list[modality_index])):
            start_positions = []
            end_positions = []
            chromosomes = []
            for gene in gene_group.split("_"):
                ensembl_gene_id_chromosome_name_start_position_end_position_mapping_filt = (
                    ensembl_gene_id_chromosome_name_start_position_end_position_mapping[
                        ensembl_gene_id_chromosome_name_start_position_end_position_mapping[
                            "Gene stable ID"
                        ]
                        == gene
                    ]
                )
                start_positions += list(
                    ensembl_gene_id_chromosome_name_start_position_end_position_mapping_filt[
                        "Gene start (bp)"
                    ]
                )
                end_positions += list(
                    ensembl_gene_id_chromosome_name_start_position_end_position_mapping_filt[
                        "Gene end (bp)"
                    ]
                )
                chromosomes += list(
                    ensembl_gene_id_chromosome_name_start_position_end_position_mapping_filt[
                        "Chromosome/scaffold name"
                    ]
                )
            genomic_marker_gene_group_vals = []
            try:
                for path_genomic_marker in paths_genomic_markers:
                    bw = pyBigWig.open(path_genomic_marker)
                    genomic_marker_val = bw.stats(
                        "chr" + chromosomes[0],
                        min(start_positions),
                        max(start_positions),
                        type=bw_type,
                    )[0]
                    genomic_marker_gene_group_vals.append(genomic_marker_val)
                genomic_marker_vals.append(genomic_marker_gene_group_vals)
            except RuntimeError:
                gene_groups_not_found_bw.append(gene_group_index)

        genomic_marker_vals = np.array(genomic_marker_vals).T
        indices_gene_groups_importances = list(
            set(range(len(graph_csv_index_list[modality_index]))) - set(gene_groups_not_found_bw)
        )

        genomic_marker_corr = []
        genomic_marker_corr_p = []
        for genomic_marker_vals_sub in genomic_marker_vals:
            genomic_marker_corr_pheno = []
            genomic_marker_corr_pheno_p = []
            for phenotype_index in range(len(gene_groups_importances[modality_index])):
                if modality_index == 0:
                    _filtered_importances_0 = gene_groups_importances[modality_index][0][
                        indices_gene_groups_importances
                    ]
                    indices_top_100 = np.where(
                        _filtered_importances_0
                        > np.quantile(_filtered_importances_0, 0.90)
                    )[0]
                    stat = scipy.stats.spearmanr(
                        genomic_marker_vals_sub[indices_top_100],
                        gene_groups_importances[modality_index][phenotype_index][
                            indices_gene_groups_importances
                        ][indices_top_100],
                    )
                else:
                    stat = scipy.stats.spearmanr(
                        genomic_marker_vals_sub,
                        gene_groups_importances[modality_index][phenotype_index][
                            indices_gene_groups_importances
                        ],
                    )

                genomic_marker_corr_pheno.append(stat[0])
                genomic_marker_corr_pheno_p.append(stat[1])
            genomic_marker_corr.append(genomic_marker_corr_pheno)
            genomic_marker_corr_p.append(genomic_marker_corr_pheno_p)

        genomic_marker_corr_modalities.append(genomic_marker_corr)
        genomic_marker_corr_p_modalities.append(genomic_marker_corr_p)

    return genomic_marker_corr_modalities, genomic_marker_corr_p_modalities


In [ ]:
from matplotlib.colors import TwoSlopeNorm

def get_custom_cmap(vmin, vmax):
    if vmin < 0:
        colors_heat = ['#c12075', '#f0f0f0', '#003754']
    else:
        colors_heat = ['#f0f0f0', '#003754']
    cmap_heat = LinearSegmentedColormap.from_list('custom_diverging', colors_heat, N=256)
    return cmap_heat

def plot_heatmap_gene_interactions_layer(features_matrix, xlabels, ylabels, part_indices, add_marker_labels, mode, figsize=(10, 10), vmin=None, vmax=None, row_linkage=None, column_linkage=None, save_name=False):
    features_matrix_mask = np.zeros_like(features_matrix)
    for n_part_index, part_index in enumerate(part_indices):
        features_matrix = np.insert(features_matrix, part_index + n_part_index * 1, np.zeros((1, features_matrix.shape[0])), axis=1)
        features_matrix_mask = np.insert(features_matrix_mask, part_index + n_part_index * 1, np.ones((1, features_matrix_mask.shape[0])), axis=1)
        xlabels = xlabels[:part_index + n_part_index * 1] + [''] + xlabels[part_index + n_part_index * 1:]
    if vmin is None:
        vmin = min(0, np.min(features_matrix))
    if vmax is None:
        vmax = np.max(features_matrix)
    midpoint = 0
    if vmin < 0:
        norm = TwoSlopeNorm(vmin=vmin, vcenter=midpoint, vmax=vmax)
        cmap_heat = get_custom_cmap(vmin, vmax)
        if row_linkage is None and column_linkage is None:
            g = sns.clustermap(features_matrix, figsize=(7, 4), norm=norm, mask=features_matrix_mask, xticklabels=xlabels, yticklabels=ylabels if add_marker_labels is True else [], cmap=cmap_heat, vmin=vmin, vmax=vmax, col_cluster=True, cbar_pos=(1.05, 0.4, 0.03, 0.2))
        else:
            g = sns.clustermap(features_matrix, figsize=(7, 4), norm=norm, mask=features_matrix_mask, xticklabels=xlabels, yticklabels=ylabels if add_marker_labels is True else [], cmap=cmap_heat, vmin=vmin, vmax=vmax, col_cluster=True, row_linkage=row_linkage, col_linkage=column_linkage, cbar_pos=(1.05, 0.4, 0.03, 0.2))
    else:
        cmap_heat = get_custom_cmap(vmin, vmax)
        if row_linkage is None and column_linkage is None:
            g = sns.clustermap(features_matrix, figsize=(7, 4), mask=features_matrix_mask, xticklabels=xlabels, yticklabels=ylabels if add_marker_labels is True else [], cmap=cmap_heat, vmin=vmin, vmax=vmax, col_cluster=True, cbar_pos=(1.0, 0.4, 0.03, 0.2))
        else:
            g = sns.clustermap(features_matrix, figsize=(7, 4), mask=features_matrix_mask, xticklabels=xlabels, yticklabels=ylabels if add_marker_labels is True else [], cmap=cmap_heat, vmin=vmin, vmax=vmax, col_cluster=True, row_linkage=row_linkage, col_linkage=column_linkage, cbar_pos=(1.0, 0.4, 0.03, 0.2))
    row_linkage = g.dendrogram_row.linkage
    column_linkage = g.dendrogram_col.linkage
    plt.xticks(rotation=90)
    g.ax_heatmap.tick_params(axis='x')
    g.ax_heatmap.tick_params(axis='y')
    if add_marker_labels is True:
        g.ax_heatmap.tick_params(axis='y')
    else:
        g.ax_heatmap.tick_params(axis='y', which='both', left=False, right=False, labelleft=False)
        g.ax_heatmap.set_ylabel('genomic markers')
    if save_name:
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.png'), dpi=500, format='png', transparent=True, bbox_inches='tight')
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.svg'), dpi=500, transparent=True, bbox_inches='tight', format='svg')
    else:
        plt.show()
    return (row_linkage, column_linkage)


In [ ]:
ensembl_gene_id_chromosome_name_start_position_end_position_mapping = (
    _get_ensembl_gene_id_chromosome_name_start_position_end_position_mapping()
)


In [ ]:
genomic_marker_corr_modalities, genomic_marker_corr_p_modalities = (
    get_corr_go_importance_genomic_markers(
        graph_csv_index_list, gene_groups_importances, paths_genomic_markers, bw_type="max"
    )
)


In [ ]:
genomic_marker_corr_modalities = np.array(genomic_marker_corr_modalities)


In [ ]:
np.save(
    f"genomic_marker_corr_modalities_neurips2021_cite_bmmc_interaction_gene_layer_restrict.npy",
    genomic_marker_corr_modalities,
)


In [ ]:
genomic_marker_corr_modalities = np.load(
    f"genomic_marker_corr_modalities_neurips2021_cite_bmmc_interaction_gene_layer_restrict.npy"
)
genomic_marker_corr_modalities_baseline = np.load(
    f"genomic_marker_corr_modalities_neurips2021_cite_bmmc_interaction_gene_layer_restrict_baseline.npy"
)


In [ ]:
genomic_marker_corr_modalities_baseline_change = (
    genomic_marker_corr_modalities - genomic_marker_corr_modalities_baseline
)


In [ ]:
genomic_marker_corr_modalities_baseline_abs_change = abs(genomic_marker_corr_modalities) - abs(
    genomic_marker_corr_modalities_baseline
)


In [ ]:
scipy.stats.mannwhitneyu(
    genomic_marker_corr_modalities.flatten(), genomic_marker_corr_modalities_baseline.flatten()
)


In [ ]:
scipy.stats.wilcoxon(
    genomic_marker_corr_modalities.flatten(), genomic_marker_corr_modalities_baseline.flatten()
)


In [ ]:
scipy.stats.wilcoxon(
    genomic_marker_corr_modalities.flatten(), genomic_marker_corr_modalities_baseline.flatten()
)


In [ ]:
scipy.stats.ks_2samp(
    genomic_marker_corr_modalities.flatten(), genomic_marker_corr_modalities_baseline.flatten()
)


In [ ]:
set_figure_params('single', 2)
fig, ax = plt.subplots()
ax.grid(False)
print(np.mean(genomic_marker_corr_modalities))
print(np.mean(genomic_marker_corr_modalities_baseline))
mean_modalities = np.mean(genomic_marker_corr_modalities)
mean_baseline = np.mean(genomic_marker_corr_modalities_baseline)
plt.hist(genomic_marker_corr_modalities.flatten(), bins=100, alpha=0.2, label=f'GOSparseVI')
plt.hist(genomic_marker_corr_modalities_baseline.flatten(), bins=100, alpha=0.2, label=f'-Interaction layer')
plt.xlabel('Spearman corr. gene groups importance - genomic markers')
plt.ylabel('prevalence')
plt.legend()
plt.savefig(os.path.join(_OUTDIR_ABS, f'neurips2021_bmmc_interaction_gene_layer_distributions.png'))
plt.savefig(os.path.join(_OUTDIR_ABS, f'neurips2021_bmmc_interaction_gene_layer_distributions.svg'), format='svg')


In [ ]:
for analysis_lab, selected_genomic_markers_labels_lab, selected_genomic_markers_positions_lab in zip(
    analysis_labs[:1], selected_genomic_markers_labels[:1], selected_genomic_markers_positions[:1]
):
    analysis_lab = analysis_lab.replace(" ", "_")
    for modality, genomic_marker_corr_modality, genomic_marker_corr_modality_baseline, genomic_marker_corr_modality_baseline_change, genomic_marker_corr_modality_baseline_abs_change in zip(
        ["expression", "protein"],
        genomic_marker_corr_modalities,
        genomic_marker_corr_modalities_baseline,
        genomic_marker_corr_modalities_baseline_change,
        genomic_marker_corr_modalities_baseline_abs_change,
    ):
        indices_top_selected_genomic_markers_positions_lab = set(
            np.where(
                np.abs(genomic_marker_corr_modality)
                > np.quantile(np.abs(genomic_marker_corr_modality[selected_genomic_markers_positions_lab]), 0.999)
            )[0]
        ) & set(selected_genomic_markers_positions_lab)
        indices_top_genomic_marker_corr_modality_baseline_change = set(
            np.where(
                np.abs(genomic_marker_corr_modality_baseline_change)
                > np.quantile(
                    np.abs(genomic_marker_corr_modality_baseline_change[selected_genomic_markers_positions_lab]), 0.999
                )
            )[0]
        ) & set(selected_genomic_markers_positions_lab)
        selected_genomic_markers_positions_top = np.array(
            list(indices_top_selected_genomic_markers_positions_lab | indices_top_genomic_marker_corr_modality_baseline_change)
        )

        unique_elements, counts = np.unique(np.array(genomic_markers)[selected_genomic_markers_positions_top], return_counts=True)
        _, first_indices = np.unique(np.array(genomic_markers)[selected_genomic_markers_positions_top], return_index=True)

        y_shape = len(np.array(selected_genomic_markers_positions_top)[first_indices]) * 0.2
        add_marker_labels = True

        row_linkage, column_linkage = plot_heatmap_gene_interactions_layer(
            genomic_marker_corr_modality[selected_genomic_markers_positions_top][first_indices],
            sorted(phenotypes_labels),
            np.array(genomic_markers)[selected_genomic_markers_positions_top][first_indices],
            [],
            add_marker_labels,
            "subset",
            figsize=(20, y_shape),
            save_name=f"neurips2021_bmmc_interaction_gene_layer_top_{analysis_lab}_{modality}_with_interaction_layer",
        )
        plot_heatmap_gene_interactions_layer(
            genomic_marker_corr_modality_baseline[selected_genomic_markers_positions_top][first_indices],
            sorted(phenotypes_labels),
            np.array(genomic_markers)[selected_genomic_markers_positions_top][first_indices],
            [],
            add_marker_labels,
            "subset",
            figsize=(20, y_shape),
            row_linkage=row_linkage,
            column_linkage=column_linkage,
            save_name=f"neurips2021_bmmc_interaction_gene_layer_top_{analysis_lab}_{modality}_baseline_without_interaction_layer",
        )
        plot_heatmap_gene_interactions_layer(
            genomic_marker_corr_modality_baseline_change[selected_genomic_markers_positions_top][first_indices],
            sorted(phenotypes_labels),
            np.array(genomic_markers)[selected_genomic_markers_positions_top][first_indices],
            [],
            add_marker_labels,
            "subset",
            figsize=(20, y_shape),
            row_linkage=row_linkage,
            column_linkage=column_linkage,
            save_name=f"neurips2021_bmmc_interaction_gene_layer_top_{analysis_lab}_{modality}_change_linkage",
        )
        plot_heatmap_gene_interactions_layer(
            genomic_marker_corr_modality_baseline_change[selected_genomic_markers_positions_top][first_indices],
            sorted(phenotypes_labels),
            np.array(genomic_markers)[selected_genomic_markers_positions_top][first_indices],
            [],
            add_marker_labels,
            "subset",
            figsize=(20, y_shape),
            save_name=f"neurips2021_bmmc_interaction_gene_layer_top_{analysis_lab}_{modality}_change",
        )
        plot_heatmap_gene_interactions_layer(
            genomic_marker_corr_modality_baseline_abs_change[selected_genomic_markers_positions_top][first_indices],
            sorted(phenotypes_labels),
            np.array(genomic_markers)[selected_genomic_markers_positions_top][first_indices],
            [],
            add_marker_labels,
            "subset",
            figsize=(20, y_shape),
            row_linkage=row_linkage,
            column_linkage=column_linkage,
            save_name=f"neurips2021_bmmc_interaction_gene_layer_top_{analysis_lab}_{modality}_abs_change_linkage",
        )
        plot_heatmap_gene_interactions_layer(
            genomic_marker_corr_modality_baseline_abs_change[selected_genomic_markers_positions_top][first_indices],
            sorted(phenotypes_labels),
            np.array(genomic_markers)[selected_genomic_markers_positions_top][first_indices],
            [],
            add_marker_labels,
            "subset",
            figsize=(20, y_shape),
            save_name=f"neurips2021_bmmc_interaction_gene_layer_top_{analysis_lab}_{modality}_abs_change",
        )


In [ ]:
italics = [
    "ENCFF596FVK",
    "ENCFF804YSZ",
    "ENCFF335JJU",
    "ENCFF135DIS",
    "ENCFF448BAC",
    "ENCFF365EJL",
    "ENCFF240UFL",
    "ENCFF612JUY",
    "ENCFF070PUZ",
    "ENCFF551KHO",
    "ENCFF227WQI",
    "ENCFF574RND",
    "ENCFF712SIB",
    "ENCFF516UUZ",
    "ENCFF461JBF",
    "ENCFF837JVM",
    "ENCFF564JOD",
    "ENCFF671HXG",
    "ENCFF370VNR",
    "ENCFF103NZN",
    "ENCFF454NLI",
    "ENCFF342DXD",
    "ENCFF773GNR",
    "ENCFF875QZF",
    "ENCFF288PSZ",
    "ENCFF329ZWA",
    "ENCFF715HYZ",
    "ENCFF124PXB",
    "ENCFF582NAG",
    "ENCFF272LUB",
    "ENCFF676UBU",
    "ENCFF243OUW",
    "ENCFF143RUK",
    "ENCFF041TQQ",
]


In [ ]:
for encff in [
    "ENCFF160ZNA",
    "ENCFF864PWN",
    "ENCFF104GDH",
    "ENCFF106EDU",
    "ENCFF530IWB",
    "ENCFF293XFC",
    "ENCFF158VSI",
    "ENCFF371VGO",
    "ENCFF949AOV",
    "ENCFF199FSE",
    "ENCFF596FVK",
    "ENCFF232TBZ",
    "ENCFF211RPU",
    "ENCFF560VXL",
    "ENCFF567RJL",
    "ENCFF842WAK",
    "ENCFF206NPF",
    "ENCFF253PNH",
    "ENCFF110ZZE",
    "ENCFF442MRI",
    "ENCFF250RIG",
    "ENCFF675JPF",
    "ENCFF985YBO",
    "ENCFF670OSB",
    "ENCFF215TUE",
    "ENCFF952ZRK",
    "ENCFF671NDS",
    "ENCFF326DQL",
    "ENCFF032ORM",
    "ENCFF934VTV",
    "ENCFF843DKK",
    "ENCFF337XFG",
    "ENCFF474CNJ",
    "ENCFF466GFK",
    "ENCFF995HPN",
    "ENCFF208ICL",
    "ENCFF747YVI",
    "ENCFF462EMG",
    "ENCFF638ZCD",
    "ENCFF335JJU",
    "ENCFF155FOH",
    "ENCFF514TNL",
    "ENCFF336UPT",
    "ENCFF425DBM",
    "ENCFF118RGH",
    "ENCFF722KTI",
    "ENCFF440CYL",
    "ENCFF135DIS",
    "ENCFF489SEA",
    "ENCFF462UOM",
    "ENCFF448BAC",
    "ENCFF935PTG",
    "ENCFF148GVM",
    "ENCFF082KZY",
    "ENCFF346YIF",
    "ENCFF483LYL",
    "ENCFF930IRA",
    "ENCFF155XQP",
    "ENCFF191CES",
    "ENCFF127RII",
    "ENCFF519ZFG",
    "ENCFF283CLF",
    "ENCFF263XFB",
    "ENCFF263YZZ",
    "ENCFF262TFX",
    "ENCFF934YZP",
    "ENCFF203IUO",
    "ENCFF223CKV",
    "ENCFF723VHQ",
    "ENCFF365EJL",
    "ENCFF209IFC",
    "ENCFF244ZVM",
    "ENCFF652ZLV",
    "ENCFF064SRL",
    "ENCFF043WON",
    "ENCFF397QRA",
    "ENCFF666PCT",
    "ENCFF286IOU",
    "ENCFF162QJJ",
    "ENCFF879EOK",
    "ENCFF704HVH",
    "ENCFF581LZY",
    "ENCFF331URE",
    "ENCFF258ASA",
    "ENCFF760GUS",
    "ENCFF657ROM",
    "ENCFF774SCK",
    "ENCFF497LNO",
    "ENCFF764MXC",
    "ENCFF016WRT",
    "ENCFF503UFO",
    "ENCFF164TRU",
    "ENCFF240UFL",
    "ENCFF655ZSU",
    "ENCFF288LPW",
    "ENCFF157RTJ",
    "ENCFF347WBJ",
    "ENCFF612JUY",
    "ENCFF070PUZ",
    "ENCFF970PUH",
    "ENCFF835GXP",
    "ENCFF817NXE",
    "ENCFF448GFY",
    "ENCFF928BKT",
    "ENCFF405ELK",
    "ENCFF636FAG",
    "ENCFF498WCM",
    "ENCFF832TMB",
    "ENCFF200NGJ",
    "ENCFF699KLZ",
    "ENCFF517EKS",
    "ENCFF847HGM",
    "ENCFF198SGR",
    "ENCFF551KHO",
    "ENCFF756UVW",
    "ENCFF114SLT",
    "ENCFF253NTK",
    "ENCFF610LRU",
    "ENCFF227WQI",
    "ENCFF501PQF",
    "ENCFF675MND",
    "ENCFF913ZJJ",
    "ENCFF373JER",
    "ENCFF750TBY",
    "ENCFF405DRC",
    "ENCFF770WTY",
    "ENCFF810ZPC",
    "ENCFF427YFE",
    "ENCFF723UZM",
    "ENCFF855QPP",
    "ENCFF131CUG",
    "ENCFF408KTS",
    "ENCFF503FPF",
    "ENCFF266ORM",
    "ENCFF521YNL",
    "ENCFF408DGX",
    "ENCFF396SRD",
    "ENCFF037QZG",
    "ENCFF691QRL",
    "ENCFF113JYQ",
    "ENCFF461WJA",
    "ENCFF574RND",
    "ENCFF806AVD",
    "ENCFF213RKS",
    "ENCFF273RVE",
    "ENCFF244MOS",
    "ENCFF128CFI",
    "ENCFF274ZIX",
    "ENCFF541CKQ",
    "ENCFF832GNG",
    "ENCFF768NOJ",
    "ENCFF921ISO",
    "ENCFF033EVS",
    "ENCFF845HTM",
    "ENCFF037VAZ",
    "ENCFF140BXD",
    "ENCFF712SIB",
    "ENCFF933LNW",
    "ENCFF336PZX",
    "ENCFF432OUZ",
    "ENCFF780JRJ",
    "ENCFF916NBA",
    "ENCFF107GXG",
    "ENCFF112BYB",
    "ENCFF375KUG",
    "ENCFF398LBP",
    "ENCFF544TIU",
    "ENCFF836FEQ",
    "ENCFF564JOD",
    "ENCFF671HXG",
    "ENCFF280QPM",
    "ENCFF643SJZ",
    "ENCFF419IBT",
    "ENCFF830KGJ",
    "ENCFF772GDY",
    "ENCFF085WAW",
    "ENCFF191VQV",
    "ENCFF463IKW",
    "ENCFF918KYJ",
    "ENCFF122XON",
    "ENCFF563WCK",
    "ENCFF021HPB",
    "ENCFF026NAA",
    "ENCFF360BJK",
    "ENCFF013FZZ",
    "ENCFF854EIV",
    "ENCFF584OLL",
    "ENCFF692MSP",
    "ENCFF548JIB",
    "ENCFF386SIX",
    "ENCFF191RLR",
    "ENCFF740VPW",
    "ENCFF585IAR",
    "ENCFF914ZIB",
    "ENCFF143PFU",
    "ENCFF198WJK",
    "ENCFF955QWG",
    "ENCFF259JPA",
    "ENCFF706ROY",
    "ENCFF516AAU",
    "ENCFF384BTU",
    "ENCFF974SRQ",
    "ENCFF386AWZ",
    "ENCFF130GMP",
    "ENCFF404COW",
    "ENCFF808EXL",
    "ENCFF260GXL",
    "ENCFF004ZMA",
    "ENCFF395BVM",
    "ENCFF407OAJ",
    "ENCFF800QUT",
    "ENCFF416PCN",
    "ENCFF818MPI",
    "ENCFF721HET",
    "ENCFF601NPD",
    "ENCFF442IDG",
    "ENCFF805EHS",
    "ENCFF511RUY",
    "ENCFF941JUQ",
    "ENCFF614TYG",
    "ENCFF735YUX",
    "ENCFF766VME",
    "ENCFF701JND",
    "ENCFF370VNR",
    "ENCFF103NZN",
    "ENCFF529QQI",
    "ENCFF431SAX",
    "ENCFF190FPU",
    "ENCFF970SIA",
    "ENCFF293QAG",
    "ENCFF022EWV",
    "ENCFF908UVY",
    "ENCFF596CNE",
    "ENCFF743DXN",
    "ENCFF685GZT",
    "ENCFF316PSV",
    "ENCFF475BKW",
    "ENCFF630LBW",
    "ENCFF794REJ",
    "ENCFF454NLI",
    "ENCFF145GDZ",
    "ENCFF796VGZ",
    "ENCFF846LZT",
    "ENCFF478OGJ",
    "ENCFF316KJE",
    "ENCFF163YNA",
    "ENCFF127BNU",
    "ENCFF098FKU",
    "ENCFF167PVP",
    "ENCFF274UBU",
    "ENCFF125SPZ",
    "ENCFF971IQQ",
    "ENCFF402ULS",
    "ENCFF495KOS",
    "ENCFF036TCU",
    "ENCFF629KBJ",
    "ENCFF060GMB",
    "ENCFF682XOQ",
    "ENCFF342DXD",
    "ENCFF494PCP",
    "ENCFF780PZN",
    "ENCFF269VGE",
    "ENCFF313CYB",
    "ENCFF129BBR",
    "ENCFF420CLL",
    "ENCFF511ZRC",
    "ENCFF039GGQ",
    "ENCFF773GNR",
    "ENCFF676NJU",
    "ENCFF332INK",
    "ENCFF897MKD",
    "ENCFF346NDO",
    "ENCFF235XFA",
    "ENCFF875QZF",
    "ENCFF508RXH",
    "ENCFF169WSC",
    "ENCFF062OZT",
    "ENCFF003NIX",
    "ENCFF405BON",
    "ENCFF988HVV",
    "ENCFF768QRC",
    "ENCFF288PSZ",
    "ENCFF854BXW",
    "ENCFF145FWH",
    "ENCFF733QSR",
    "ENCFF199WRR",
    "ENCFF547AGX",
    "ENCFF403AZP",
    "ENCFF364OGY",
    "ENCFF106ZRO",
    "ENCFF325ACI",
    "ENCFF807KVX",
    "ENCFF594LZQ",
    "ENCFF422IWO",
    "ENCFF831QAC",
    "ENCFF163JPZ",
    "ENCFF607UFI",
    "ENCFF615VLU",
    "ENCFF271CFD",
    "ENCFF495YNJ",
    "ENCFF705BCZ",
    "ENCFF856CAV",
    "ENCFF512NFV",
    "ENCFF132HZU",
    "ENCFF306XYI",
    "ENCFF064AVI",
    "ENCFF473UBB",
    "ENCFF573MHC",
    "ENCFF690ABT",
    "ENCFF226VDW",
    "ENCFF655SMR",
    "ENCFF744MAU",
    "ENCFF715HYZ",
    "ENCFF843FWI",
    "ENCFF124PXB",
    "ENCFF431FTN",
    "ENCFF389PED",
    "ENCFF547OVU",
    "ENCFF865XSE",
    "ENCFF391FZA",
    "ENCFF582NAG",
    "ENCFF272LUB",
    "ENCFF631RZW",
    "ENCFF563XMY",
    "ENCFF551YRA",
    "ENCFF770ATI",
    "ENCFF676UBU",
    "ENCFF377DDT",
    "ENCFF223RME",
    "ENCFF454RXI",
    "ENCFF243OUW",
    "ENCFF438XUP",
    "ENCFF843WTJ",
    "ENCFF552HWW",
    "ENCFF798IJB",
    "ENCFF331LJE",
    "ENCFF703CTG",
    "ENCFF430RCW",
    "ENCFF143RUK",
    "ENCFF366SMI",
    "ENCFF230DRI",
    "ENCFF304UOP",
    "ENCFF565PQQ",
    "ENCFF830VNI",
    "ENCFF184MJA",
    "ENCFF183GAS",
    "ENCFF974KEW",
    "ENCFF200DLV",
    "ENCFF713BBY",
    "ENCFF684UYV",
    "ENCFF913TWG",
    "ENCFF674URK",
    "ENCFF650DYQ",
    "ENCFF979OJR",
    "ENCFF600BUX",
    "ENCFF816TPI",
    "ENCFF264SMF",
    "ENCFF834QAV",
    "ENCFF041TQQ",
    "ENCFF598RUP",
    "ENCFF152HYG",
    "ENCFF854POL",
    "ENCFF794BGU",
    "ENCFF358XWW",
    "ENCFF483RAR",
    "ENCFF550YIC",
    "ENCFF265WDL",
    "ENCFF034GGJ",
]:
    if encff in italics:
        print("\\textit{" + encff + "}")
    else:
        print(encff)


# CpG islands

In [ ]:
! wget -O cpg_islands_hg38.bed.gz "https://hgdownload.soe.ucsc.edu/goldenPath/hg38/database/cpgIslandExt.txt.gz"

! zcat cpg_islands_hg38.bed.gz | awk '{print $2"\t"$3"\t"$4}' > cpg_islands_hg38.bed

! head -3 cpg_islands_hg38.bed
! wc -l cpg_islands_hg38.bed


In [ ]:
ensembl_gene_id_chromosome_name_start_position_end_position_mapping = (
    _get_ensembl_gene_id_chromosome_name_start_position_end_position_mapping()
)


In [ ]:
coord_map = ensembl_gene_id_chromosome_name_start_position_end_position_mapping

def get_tad_coords(tad_id, coord_map):
    start_positions = []
    end_positions = []
    chromosomes = []
    for gene in tad_id.split("_"):
        gene_row = coord_map[coord_map["Gene stable ID"] == gene]
        start_positions += list(gene_row["Gene start (bp)"])
        end_positions += list(gene_row["Gene end (bp)"])
        chromosomes += list(gene_row["Chromosome/scaffold name"])
    if not chromosomes:
        return None
    return "chr" + chromosomes[0], min(start_positions), max(end_positions)

tad_ids_ordered = list(graph_csv_index_list[0])
mean_gain_per_tad = np.mean(
    np.array(
        [
            gene_groups_importances[0][i] - gene_groups_baseline_importances[0][i]
            for i in range(len(gene_groups_importances[0]))
        ]
    ),
    axis=0,
)


In [ ]:
import subprocess
import tempfile
import os
import numpy as np
import scipy.stats
import matplotlib.pyplot as plt
from tqdm import tqdm
import shutil
BEDTOOLS = shutil.which("bedtools") or "bedtools"
CPG_BED = 'cpg_islands_hg38.bed'

def count_cpg_overlap(chrom, start, end, cpg_bed, bedtools=BEDTOOLS):
    tmp = f'/tmp/cpg_query_{os.getpid()}.bed'
    with open(tmp, 'w') as f:
        f.write(f'{chrom}\t{start}\t{end}\n')
    cmd = f'{bedtools} intersect -a {tmp} -b {cpg_bed} -wo'
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    os.remove(tmp)
    lines = [l for l in result.stdout.strip().split('\n') if l]
    n_overlaps = len(lines)
    length_kb = max(1, (end - start) / 1000)
    return n_overlaps / length_kb
cpg_density_per_tad = []
failed_tads = 0
for tad_id in tqdm(tad_ids_ordered):
    coords = get_tad_coords(tad_id, coord_map)
    if coords is None:
        cpg_density_per_tad.append(np.nan)
        failed_tads += 1
        continue
    chrom, start, end = coords
    if not chrom.replace('chr', '').isdigit() and chrom not in ['chrX', 'chrY']:
        cpg_density_per_tad.append(np.nan)
        failed_tads += 1
        continue
    density = count_cpg_overlap(chrom, start, end, CPG_BED)
    cpg_density_per_tad.append(density)
cpg_density_per_tad = np.array(cpg_density_per_tad)
print(f'Failed/skipped TADs: {failed_tads} / {len(tad_ids_ordered)}')
print(f'TADs with CpG data:  {np.sum(~np.isnan(cpg_density_per_tad))}')
print(f'Non-zero entries:    {np.sum(cpg_density_per_tad > 0)}')
print(f'Min: {cpg_density_per_tad[~np.isnan(cpg_density_per_tad)].min():.4f}, Max: {cpg_density_per_tad[~np.isnan(cpg_density_per_tad)].max():.4f}')
assert len(cpg_density_per_tad) == len(mean_gain_per_tad), f'Length mismatch: cpg={len(cpg_density_per_tad)}, gain={len(mean_gain_per_tad)}'
print('Length check passed.')
valid = ~np.isnan(cpg_density_per_tad)
rho_cpg, p_cpg = scipy.stats.spearmanr(cpg_density_per_tad[valid], mean_gain_per_tad[valid])
rng = np.random.default_rng(42)
boots_cpg = []
idx_valid = np.where(valid)[0]
for _ in range(10000):
    idx = rng.choice(len(idx_valid), len(idx_valid), replace=True)
    boots_cpg.append(scipy.stats.spearmanr(cpg_density_per_tad[idx_valid[idx]], mean_gain_per_tad[idx_valid[idx]]).statistic)
ci_lo_cpg = np.percentile(boots_cpg, 2.5)
ci_hi_cpg = np.percentile(boots_cpg, 97.5)
print(f'\nCpG density vs TAD importance gain:')
print(f'  Spearman rho = {rho_cpg:.3f}, p = {p_cpg:.4e}')
print(f'  Bootstrap 95% CI: [{ci_lo_cpg:.3f}, {ci_hi_cpg:.3f}]')
set_figure_params('single', 2.2)
fig, ax = plt.subplots()
ax.scatter(cpg_density_per_tad[valid], mean_gain_per_tad[valid], alpha=0.4, s=3, color='#70acc0')
ax.set_xlabel('CpG island density (islands per kb)')
ax.set_ylabel('Mean TAD importance gain')
ax.set_title(f'Spearman ρ = {rho_cpg:.3f}, p = {p_cpg:.3e}')
plt.tight_layout()
plt.savefig(os.path.join(_OUTDIR_ABS, 'cpg_density_vs_tad_importance_gain.pdf'), bbox_inches='tight')
plt.savefig(os.path.join(_OUTDIR_ABS, 'cpg_density_vs_tad_importance_gain.svg'), bbox_inches='tight', format='svg')
plt.show()
